# Wind Power Forecast Accuracy Analysis

## Objective
This notebook analyzes the accuracy characteristics of wind power generation forecasts. The goal is to understand:
1. Error distribution and statistics (mean, median, p99)
2. How forecast accuracy varies with horizon length
3. Temporal patterns in forecast errors
4. Reliability of wind power predictions for electricity demand planning

## Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import requests
import json

# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Data Loading

Load actual and forecast wind generation data from the Elexon API endpoints.

In [ ]:
# Configuration
START_DATE = datetime(2025, 1, 15)
END_DATE = datetime(2025, 1, 31)
API_KEY = ''  # Add your API key here

# API Endpoints
ACTUAL_API = 'https://bmrs.elexon.co.uk/api-documentation/endpoint/datasets/FUELHH/stream'
FORECAST_API = 'https://bmrs.elexon.co.uk/api-documentation/endpoint/datasets/WINDFOR/stream'

def fetch_actual_data(start_date, end_date):
    """
    Fetch actual wind generation data
    """
    params = {
        'startTime': start_date.isoformat(),
        'endTime': end_date.isoformat(),
        'fuelType': 'WIND'
    }
    
    try:
        response = requests.get(ACTUAL_API, params=params)
        data = response.json()
        df = pd.DataFrame(data)
        df['startTime'] = pd.to_datetime(df['startTime'])
        df = df.sort_values('startTime')
        print(f"Loaded {len(df)} actual generation records")
        return df
    except Exception as e:
        print(f"Error loading actual data: {e}")
        return pd.DataFrame()

def fetch_forecast_data(start_date, end_date):
    """
    Fetch wind generation forecast data
    """
    params = {
        'startTime': start_date.isoformat(),
        'endTime': end_date.isoformat()
    }
    
    try:
        response = requests.get(FORECAST_API, params=params)
        data = response.json()
        df = pd.DataFrame(data)
        df['startTime'] = pd.to_datetime(df['startTime'])
        df['publishTime'] = pd.to_datetime(df['publishTime'])
        df = df.sort_values('startTime')
        print(f"Loaded {len(df)} forecast records")
        return df
    except Exception as e:
        print(f"Error loading forecast data: {e}")
        return pd.DataFrame()

# Load data
actual_df = fetch_actual_data(START_DATE, END_DATE)
forecast_df = fetch_forecast_data(START_DATE, END_DATE)

print(f"\nActual data shape: {actual_df.shape}")
print(f"Forecast data shape: {forecast_df.shape}")

## 2. Data Preparation and Cleaning

In [ ]:
# Display sample data
print("Actual data sample:")
print(actual_df.head())
print("\nForecast data sample:")
print(forecast_df.head())

# Check for missing values
print("\nMissing values in actual data:")
print(actual_df.isnull().sum())
print("\nMissing values in forecast data:")
print(forecast_df.isnull().sum())

In [ ]:
# Remove records with missing generation values
actual_df = actual_df.dropna(subset=['generation'])
forecast_df = forecast_df.dropna(subset=['generation'])

# Calculate forecast horizon (in hours)
forecast_df['forecast_horizon_hours'] = (forecast_df['startTime'] - forecast_df['publishTime']).dt.total_seconds() / 3600

# Filter for valid forecast horizons (0-48 hours)
forecast_df = forecast_df[(forecast_df['forecast_horizon_hours'] >= 0) & 
                          (forecast_df['forecast_horizon_hours'] <= 48)]

print(f"After cleaning:")
print(f"Actual records: {len(actual_df)}")
print(f"Forecast records: {len(forecast_df)}")
print(f"\nForecast horizon statistics:")
print(forecast_df['forecast_horizon_hours'].describe())

## 3. Error Analysis

Calculate forecast errors and key statistics.

In [ ]:
# Merge actual and forecast data by matching timestamps
merged_df = pd.merge(
    actual_df[['startTime', 'generation']].rename(columns={'generation': 'actual_generation'}),
    forecast_df.groupby('startTime')['generation'].mean().reset_index().rename(columns={'generation': 'forecast_generation'}),
    on='startTime',
    how='inner'
)

print(f"Matched records: {len(merged_df)}")
print("\nSample of merged data:")
print(merged_df.head(10))

In [ ]:
# Calculate errors
merged_df['absolute_error'] = np.abs(merged_df['actual_generation'] - merged_df['forecast_generation'])
merged_df['percentage_error'] = (merged_df['absolute_error'] / merged_df['actual_generation']) * 100
merged_df['signed_error'] = merged_df['forecast_generation'] - merged_df['actual_generation']

# Display error statistics
print("=" * 60)
print("OVERALL FORECAST ERROR STATISTICS")
print("=" * 60)

mean_error = merged_df['absolute_error'].mean()
median_error = merged_df['absolute_error'].median()
p99_error = merged_df['absolute_error'].quantile(0.99)
std_error = merged_df['absolute_error'].std()
max_error = merged_df['absolute_error'].max()

print(f"Mean Absolute Error (MAE):    {mean_error:,.0f} MW")
print(f"Median Absolute Error:         {median_error:,.0f} MW")
print(f"P99 Absolute Error:            {p99_error:,.0f} MW")
print(f"Standard Deviation:            {std_error:,.0f} MW")
print(f"Maximum Error:                 {max_error:,.0f} MW")

print(f"\nMean Percentage Error:         {merged_df['percentage_error'].mean():.2f}%")
print(f"Median Percentage Error:       {merged_df['percentage_error'].median():.2f}%")
print(f"\nForecast Bias (Mean):          {merged_df['signed_error'].mean():,.0f} MW")
print(f"Forecast Bias (% of actual):   {(merged_df['signed_error'].mean() / merged_df['actual_generation'].mean()) * 100:.2f}%")

## 4. Error by Forecast Horizon

Analyze how error changes with forecast lead time.

In [ ]:
# Merge with horizon data
merged_with_horizon = pd.merge(
    merged_df,
    forecast_df.groupby('startTime')['forecast_horizon_hours'].mean().reset_index(),
    on='startTime',
    how='inner'
)

# Create horizon bins for analysis
merged_with_horizon['horizon_bin'] = pd.cut(
    merged_with_horizon['forecast_horizon_hours'],
    bins=[0, 6, 12, 24, 36, 48],
    labels=['0-6h', '6-12h', '12-24h', '24-36h', '36-48h']
)

# Calculate error by horizon
error_by_horizon = merged_with_horizon.groupby('horizon_bin', observed=True).agg({
    'absolute_error': ['mean', 'median', lambda x: x.quantile(0.99)],
    'percentage_error': 'mean',
    'startTime': 'count'
}).round(2)

error_by_horizon.columns = ['Mean Error (MW)', 'Median Error (MW)', 'P99 Error (MW)', 'Mean % Error', 'Count']

print("\n" + "=" * 80)
print("ERROR BY FORECAST HORIZON")
print("=" * 80)
print(error_by_horizon)

## 5. Temporal Analysis

Analyze how forecast errors vary by time of day.

In [ ]:
# Extract time of day
merged_df['hour'] = merged_df['startTime'].dt.hour
merged_df['day_of_week'] = merged_df['startTime'].dt.day_name()

# Error by hour of day
error_by_hour = merged_df.groupby('hour').agg({
    'absolute_error': ['mean', 'std', lambda x: x.quantile(0.99)],
    'percentage_error': 'mean',
    'actual_generation': 'mean',
    'startTime': 'count'
}).round(2)

error_by_hour.columns = ['Mean Error', 'Std Dev', 'P99 Error', 'Mean % Error', 'Avg Generation', 'Count']

print("\n" + "=" * 100)
print("ERROR BY HOUR OF DAY")
print("=" * 100)
print(error_by_hour)

## 6. Visualizations

In [ ]:
# Figure 1: Error Distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Histogram of absolute errors
axes[0, 0].hist(merged_df['absolute_error'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(mean_error, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_error:,.0f} MW')
axes[0, 0].axvline(median_error, color='green', linestyle='--', linewidth=2, label=f'Median: {median_error:,.0f} MW')
axes[0, 0].set_xlabel('Absolute Error (MW)', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Distribution of Forecast Errors', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(axis='y', alpha=0.3)

# Percentage error distribution
axes[0, 1].hist(merged_df['percentage_error'], bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Percentage Error (%)', fontsize=11, fontweight='bold')
axes[0, 1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Distribution of Percentage Errors', fontsize=12, fontweight='bold')
axes[0, 1].grid(axis='y', alpha=0.3)

# Box plot by horizon
merged_with_horizon.boxplot(column='absolute_error', by='horizon_bin', ax=axes[1, 0])
axes[1, 0].set_xlabel('Forecast Horizon', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Absolute Error (MW)', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Error Distribution by Forecast Horizon', fontsize=12, fontweight='bold')
plt.sca(axes[1, 0])
plt.xticks(rotation=45)

# Actual vs Forecast scatter
axes[1, 1].scatter(merged_df['actual_generation'], merged_df['forecast_generation'], 
                   alpha=0.5, s=20, color='steelblue')
max_val = max(merged_df['actual_generation'].max(), merged_df['forecast_generation'].max())
axes[1, 1].plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect Forecast')
axes[1, 1].set_xlabel('Actual Generation (MW)', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Forecast Generation (MW)', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Actual vs Forecast Generation', fontsize=12, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Figure 1: Error Analysis Visualizations")

In [ ]:
# Figure 2: Error by Forecast Horizon
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

horizon_stats = merged_with_horizon.groupby('horizon_bin', observed=True).agg({
    'absolute_error': ['mean', 'std'],
    'percentage_error': 'mean'
}).round(2)

# Mean error by horizon
mean_errors = horizon_stats[('absolute_error', 'mean')]
std_errors = horizon_stats[('absolute_error', 'std')]

axes[0].bar(range(len(mean_errors)), mean_errors, yerr=std_errors, 
            capsize=5, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].set_xticks(range(len(mean_errors)))
axes[0].set_xticklabels(mean_errors.index, rotation=45)
axes[0].set_ylabel('Absolute Error (MW)', fontsize=11, fontweight='bold')
axes[0].set_title('Mean Absolute Error vs Forecast Horizon', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Percentage error by horizon
pct_errors = horizon_stats[('percentage_error', 'mean')]
axes[1].plot(range(len(pct_errors)), pct_errors, marker='o', linewidth=2.5, 
             markersize=8, color='coral')
axes[1].set_xticks(range(len(pct_errors)))
axes[1].set_xticklabels(pct_errors.index, rotation=45)
axes[1].set_ylabel('Mean Percentage Error (%)', fontsize=11, fontweight='bold')
axes[1].set_title('Percentage Error vs Forecast Horizon', fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Figure 2: Error Degradation with Forecast Horizon")

In [ ]:
# Figure 3: Temporal Patterns
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

hour_mean_error = merged_df.groupby('hour')['absolute_error'].mean()
hour_mean_generation = merged_df.groupby('hour')['actual_generation'].mean()

# Error by hour
axes[0].plot(hour_mean_error.index, hour_mean_error.values, marker='o', linewidth=2.5, 
             markersize=8, color='steelblue')
axes[0].fill_between(hour_mean_error.index, hour_mean_error.values, alpha=0.3, color='steelblue')
axes[0].set_xlabel('Hour of Day', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Mean Absolute Error (MW)', fontsize=11, fontweight='bold')
axes[0].set_title('Forecast Error by Time of Day', fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3)
axes[0].set_xticks(range(0, 24, 2))

# Generation by hour
axes[1].plot(hour_mean_generation.index, hour_mean_generation.values, marker='s', linewidth=2.5, 
             markersize=8, color='green')
axes[1].fill_between(hour_mean_generation.index, hour_mean_generation.values, alpha=0.3, color='green')
axes[1].set_xlabel('Hour of Day', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Average Generation (MW)', fontsize=11, fontweight='bold')
axes[1].set_title('Average Wind Generation by Time of Day', fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3)
axes[1].set_xticks(range(0, 24, 2))

plt.tight_layout()
plt.show()

print("Figure 3: Temporal Patterns in Forecast Errors")

## 7. Conclusions and Recommendations

Based on the comprehensive analysis of wind power forecast accuracy:

In [ ]:
print("\n" + "="*80)
print("FORECAST ACCURACY ANALYSIS - KEY FINDINGS")
print("="*80)

print("\n1. OVERALL ACCURACY METRICS")
print("-" * 80)
print(f"   • Mean Absolute Error: {mean_error:,.0f} MW")
print(f"   • Median Absolute Error: {median_error:,.0f} MW")
print(f"   • 99th Percentile Error: {p99_error:,.0f} MW")
print(f"   • Forecast Bias: {merged_df['signed_error'].mean():,.0f} MW ({(merged_df['signed_error'].mean() / merged_df['actual_generation'].mean()) * 100:.2f}% of mean)")

# Reliability assessment
actual_avg = merged_df['actual_generation'].mean()
print(f"\n2. RELIABILITY FOR ELECTRICITY DEMAND PLANNING")
print("-" * 80)
print(f"   Average wind generation: {actual_avg:,.0f} MW")
print(f"   Error margin (±1 MAE): ±{mean_error:,.0f} MW ({(mean_error/actual_avg)*100:.1f}% of average)")
print(f"   Worst case (P99): ±{p99_error:,.0f} MW ({(p99_error/actual_avg)*100:.1f}% of average)")

# Safety margin recommendation
reliable_capacity = actual_avg - p99_error
print(f"\n3. SAFE CAPACITY PLANNING")
print("-" * 80)
print(f"   Based on P99 error margin:")
print(f"   • Recommended reliable capacity: {max(0, reliable_capacity):,.0f} MW")
print(f"   • This represents {(max(0,reliable_capacity)/actual_avg)*100:.1f}% of average generation")
print(f"   • Safety margin: {p99_error:,.0f} MW")

# Horizon impact
print(f"\n4. IMPACT OF FORECAST HORIZON")
print("-" * 80)
error_by_horizon_mean = merged_with_horizon.groupby('horizon_bin', observed=True)['absolute_error'].mean()
for idx, (horizon, error) in enumerate(error_by_horizon_mean.items()):
    if idx == 0:
        print(f"   Short-term ({horizon}): {error:,.0f} MW")
    else:
        improvement = error_by_horizon_mean.iloc[0] - error
        print(f"   {horizon}: {error:,.0f} MW (degradation from short-term)")

print(f"\n5. RECOMMENDATIONS FOR ELECTRICITY DEMAND PLANNING")
print("-" * 80)
print(f"   ✓ For short-term planning (0-6 hours): Use forecasted values with ±{mean_error:,.0f} MW margin")
print(f"   ✓ For medium-term planning (6-24 hours): Use ±{merged_with_horizon[merged_with_horizon['horizon_bin']=='12-24h']['absolute_error'].mean():,.0f} MW margin")
print(f"   ✓ Reserve capacity: {max(0, reliable_capacity):,.0f} MW for reliable base load")
print(f"   ✓ Ensure backup generation > {p99_error:,.0f} MW for unexpected forecast variations")
print("\n" + "="*80)